# Homología Persistente

En esta libreta, calcularemos la homología persistente de algunas filtraciones.

Empezaremos instalando algunas librerías que nos haran falta.

In [ ]:
%%capture
pip install gudhi networkx trimesh scipy ot

Por otro lado, vamos a instalar el módulo PHAT (https://www.sciencedirect.com/science/article/pii/S0747717116300098) que nos permitirá trabajar con las matrices sobre $\mathbb{Z}_2$. Para esto utilizaremos una versión algo modificada.

In [ ]:
# %%capture
# !pip install setuptools pybind11
# !pip install --no-build-isolation git+https://bitbucket.org/atorras1618/phat.git

Como seguramente estaremos ejecutando la libreta en Google Colab, ejecutamos la siguiente celda para importar funciones auxiliares para trabajar con los laboratorios.

In [ ]:
%%capture
import os

# Check if we are running in Colab
if 'google.colab' in str(get_ipython()):
    repo_name = 'MyST'
    if not os.path.exists(repo_name):
        !git clone https://github.com/atorras1618/{repo_name}.git
    
    os.chdir(repo_name)
    import sys
    sys.path.append(os.getcwd())

import funciones_auxiliares

## Part 1: Filtraciones de estrella inferior

En diversas situaciones, uno dispone de una triangulación sin tener una filtración natural impuesta por los datos. O quizás uno prefiere calcular una filtración dada por la disposición espacial de la triangulación. En este contexto, la filtración de estrella inferior se presenta como una filtración muy natural. En la siguiente figura, podemos ver la triangulación de una figura conocida, el "conejo de Stanford", con la filtración por subniveles de estrella inferior determinada por la dirección `(1,1,0)`. A continuación, veremos como construir esta filtración y calcularemos su homología persistente.

![Bunny](images/bunny_sublevel.png)

Empezaremos descargando la triangulación del conejo de Stanford, si aún no lo hemos hecho. Durante esta práctica, utilizaremos una triangulación que utiliza menos triángulos que el objeto original, esta reducción se encuentra en la librería `libigl` y lo obtendremos mediante su URL.

In [ ]:
import urllib.request
import os
import trimesh

# empezamos descargando la triangulación de una forma conocida como el "conejo de Stanford"
# Usamos una reducción de la triangulación presente in el siguiente URL:
url = "https://raw.githubusercontent.com/libigl/libigl-tutorial-data/master/bunny.off"
filename = "class_bunny.off"
if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)
# obtenemos los vértices y triángulos de la triangulación
mesh = trimesh.load(filename)
vertices = mesh.vertices
triangles = mesh.faces 

In [ ]:
import gudhi
st_bunny = gudhi.SimplexTree()

In [ ]:
import numpy as np
st_bunny.insert_batch(triangles.transpose(), np.zeros(triangles.shape[0]))

In [ ]:
import matplotlib.pyplot as plt
from funciones_auxiliares import plot_simplex_tree_3D

plot_simplex_tree_3D(st_bunny, vertices, alpha_faces=0.7)
# ajustamos la cámara mediante el siguiente comando
plt.gca().view_init(elev=100, azim=-90)
plt.tight_layout()
plt.show()

Como en la libreta anterior, podemos inspeccionar la información del complejo simplicial `st_bunny`, incluyendo los números de betti. Para capturar $\beta_2$, aumentamos la dimensión del complejo simplicial a $3$.

In [ ]:
st_bunny.set_dimension(3)
print("Información sobre el complejo simplicial bunny:")
print(f"Dimensión: {st_bunny.dimension()}")
print(f"Número de vértices: {st_bunny.num_vertices()}")
print(f"Número de símplices: {st_bunny.num_simplices()}")
st_bunny.compute_persistence()
print(f"Números de Betti: {st_bunny.betti_numbers()}")

In [ ]:
from funciones_auxiliares import height_filtration_from_mesh

st_bunny = height_filtration_from_mesh(mesh, direction=[1,1,0])

In [ ]:
plot_simplex_tree_3D(st_bunny, vertices, alpha_faces=0.7)
# ajustamos la cámara mediante el siguiente comando
plt.gca().view_init(elev=100, azim=-90)
plt.tight_layout()
plt.show()

In [ ]:
from funciones_auxiliares import diccionario_simplices
import os


num_slices = 5
first_sublevel = 0.01
shift = 0.045
fig = plt.figure(figsize=(num_slices*5, 6))
for idx in range(num_slices):
    ax = fig.add_subplot(1, num_slices, idx + 1, projection='3d')
    st_aux = st_bunny.copy()
    st_aux.prune_above_filtration(first_sublevel+idx*shift)
    plot_simplex_tree_3D(st_aux, vertices, alpha_faces=0.7, ax=ax)
    ax.view_init(elev=100, azim=-90)
    st_aux.set_dimension(3)
    st_aux.compute_persistence()
    ax.set_title(f"Betti: {st_aux.betti_numbers()}", fontsize=20)
plt.tight_layout()
plt.savefig(os.path.join("images", "bunny_sublevel.png"), dpi=50)

Como podemos comprobar, los números de Betti cambian con el valor máximo de filtración que tomamos para el subcomplejo de nivel. 

También podemos obtener directamente los números de betti persistentes entre dos valores de filtración $a < b$ en $\mathbb{R}$
$$
\beta_p^{a,b} = \textrm{im} (h^{a,b}_p \colon H_p(K_a) \rightarrow H_p(K_b))
$$
para una filtración $K_r$ indexada por $r \in \mathbb{R}$. Por ejemplo, a continuación calculamos $\beta_1^{0.09, 0.12}$.

In [ ]:
st_bunny.set_dimension(3)
st_bunny.compute_persistence()
st_bunny.persistent_betti_numbers(0.09, 0.12)

In [ ]:
import gudhi
diag = st_bunny.persistence()
gudhi.plot_persistence_barcode(diag)
impath = os.path.join("images", "pbarcode-bunny.png")
plt.savefig(impath, dpi=100)

In [ ]:
st_bunny.lower_star_persistence_generators()

In [ ]:

plot_simplex_tree_3D(st_bunny, vertices, plot_lower_star_generators=True)
# ajustamos la cámara mediante el siguiente comando
plt.gca().view_init(elev=100, azim=-90)
plt.tight_layout()
plt.legend()
impath = os.path.join("images", "crit-points-bunny.png")
plt.savefig(impath, dpi=100)

In [ ]:
ax = gudhi.plot_persistence_diagram(diag)
# We can modify the title, aspect, etc.
ax.set_title("Persistence diagram of a torus")
ax.set_aspect("equal")  # forces to be square shaped
impath = os.path.join("images", "pdiag-bunny.png")
plt.savefig(impath, dpi=100)

Podemos leer exactamente que simplices han sido emparejados por el algoritmo.

In [ ]:
st_bunny.persistence_pairs()

# Calculos matriciales, representantes

In [ ]:
T = gudhi.SimplexTree()
T.insert([0,1,2,3])
T.remove_maximal_simplex([0,1,2,3])
for simplex, filt in T.get_simplices():
    T.assign_filtration(simplex, max(simplex))

In [ ]:
from funciones_auxiliares import diferenciales
import sympy as sp

list_dif = diferenciales(T)
for dim in range(T.dimension()+1):
    print(f"Diferencial en dimensión {dim}:")
    print()
    sp.pprint(list_dif[dim])
    print()

In [ ]:
T.compute_persistence()

In [ ]:
T.persistence_pairs()

In [ ]:
from funciones_auxiliares import column_reduce_DV

In [ ]:
from sympy import pprint

D = list_dif[1]
R, V = column_reduce_DV(D)
print(f"R = D · V, in dimension {1}")
pprint(R)
print("=")
pprint(D)
print("·")
pprint(V)

In [ ]:
Rt = R.transpose()
num_rows, num_cols = Rt.shape
pivot_row = 0
pivot_index = -1
for r in range(pivot_row, num_rows):
    if Rt[r, 0] != 0:
        pivot_index = r
        break

pivot_index

In [ ]:
R, V = column_reduce_DV(list_dif[2])

In [ ]:
R

In [ ]:
list_dif[2]*V

In [ ]:
V

### Bottleneck Stability

Compute bottleneck distance with respect to another direction.

In [ ]:
st_bunny_2 = height_filtration_from_mesh(mesh, direction=[1.1,0.9,0])
st_bunny_2.set_dimension(3)
st_bunny_2.compute_persistence()
diag_dim_1 = st_bunny.persistence_intervals_in_dimension(1)
diag_2_dim_1 = st_bunny_2.persistence_intervals_in_dimension(1)
gudhi.bottleneck_distance(diag_dim_1, diag_2_dim_1)

In [ ]:
# 